# Xử lý dữ liệu CO2 từ dữ liệu NetCDF4 chuyển sang dữ liệu CSV tại Việt Nam
Đọc file NetCDF4 chứa dữ liệu CO2, lọc các điểm nằm trong lãnh thổ Việt Nam và chỉ giữ lại dữ liệu có chất lượng tốt chuyển sang file CSV


In [1]:
import netCDF4 # Thư viện để đọc file NetCDF4
from netCDF4 import num2date # Hàm chuyển đổi thời gian từ NetCDF
import pandas as pd # Thư viện để xử lý dữ liệu dạng bảng
import geopandas as gpd # Thư viện để xử lý dữ liệu địa lý
from datetime import datetime, timedelta # Thư viện để xử lý thời gian
import numpy as np # Thư viện tính toán số học
import os # Thư viện làm việc với hệ thống file
import glob # Thư viện để tìm kiếm file theo mẫu



In [ ]:
# Ranh giới shapefile Việt Nam dùng để lọc các CO2 trong lãnh thổ Việt Nam
shapefile_path = "E:\\RanhGioi\\VNM_adm\\gadm41_VNM_0.shp" 

In [ ]:
def process_single_nc_file(file_path, output_folder, shapefile_path):
    """
    Xử lý một file NetCDF4, lọc dữ liệu CO2 cho Việt Nam với chất lượng tốt
    
    Tham số:
        file_path: Đường dẫn đến file .nc4 cần xử lý
        output_folder: Thư mục lưu file CSV kết quả
        shapefile_path: Đường dẫn đến shapefile ranh giới Việt Nam
    """

    # Bước 1: Mở file NetCDF4
    with netCDF4.Dataset(file_path, 'r') as nc_file:

        # Bước 2: Đọc các biến cần thiết từ file NetCDF4
        required_variables = [
            'latitude',           # Vĩ độ
            'longitude',          # Kinh độ
            'time',              # Thời gian đo
            'date',              # Ngày đo
            'xco2_quality_flag', # Cờ chất lượng (0 = tốt, khác 0 = không tốt)
            'xco2'               # Nồng độ CO2 (ppm)
        ]
        ## Lấy danh sách biến có trong file
        available_variables = nc_file.variables.keys()

        ## Các biến cần thiết không có trong file
        latitude = nc_file.variables['latitude'][:]  # Mảng vĩ độ
        longitude = nc_file.variables['longitude'][:]  # Mảng kinh độ
        time_var = nc_file.variables['time']  # Biến thời gian (có metadata)
        time_data = time_var[:]  # Dữ liệu thời gian (số)
        date_data = nc_file.variables['date'][:]  # Dữ liệu ngày (năm, tháng, ngày)
        xco2_quality_flag_data = nc_file.variables['xco2_quality_flag'][:]  # Cờ chất lượng
        xco2_data = nc_file.variables['xco2'][:]  # Nồng độ CO2
        
        # Bước 3: Tạo tên file CSV đầu ra dựa trên ngày trong dữ liệu hoặc tên file gốc
        output_csv_name = None
        if date_data.shape[0] > 0 and date_data.ndim >= 2 and date_data.shape[1] >= 3:
            ## Lấy năm, tháng, ngày từ dòng đầu tiên của dữ liệu
            year_fn = int(date_data[0, 0])
            month_fn = int(date_data[0, 1])
            day_fn = int(date_data[0, 2])
            ## Tạo tên file theo định dạng oco_data_YY-MM-DD.csv
            date_for_filename = f"{str(year_fn)[-2:]}-{month_fn:02d}-{day_fn:02d}"
            output_csv_name = f"oco_data_{date_for_filename}.csv"
        else:
            ## Nếu không có dữ liệu ngày tháng, sử dụng tên file gốc
            base_name = os.path.splitext(os.path.basename(file_path))[0]
            output_csv_name = f"{base_name}_processed_quality_filtered.csv"
        ## Đường dẫn file CSV đầu ra
        output_csv_path = os.path.join(output_folder, output_csv_name)


        # Bước 4: Chuyển đổi thời gian từ giá trị số sang datetime
        time_units = getattr(time_var, 'units', 'seconds since 1970-01-01 00:00:00')
        datetime_values_flat = [] ## Danh sách lưu giá trị datetime sau chuyển đổi
        flat_time_data = time_data.flatten() ## Dữ liệu thời gian dạng phẳng

        if 'since' in time_units:
            try:
                ## Tách chuỗi để lấy thời điểm gốc
                time_origin_str_full = time_units.split(' since ')[-1]
                time_origin = datetime.strptime(time_origin_str_full, '%Y-%m-%d %H:%M:%S')
                ## Chuyển đổi từng giá trị thời gian sang datetime
                time_deltas_flat = []
                if 'hours' in time_units.lower():
                    time_deltas_flat = [timedelta(hours=float(t)) for t in flat_time_data]
                elif 'minutes' in time_units.lower():
                    time_deltas_flat = [timedelta(minutes=float(t)) for t in flat_time_data]
                elif 'days' in time_units.lower():
                    time_deltas_flat = [timedelta(days=float(t)) for t in flat_time_data]
                else:
                    time_deltas_flat = [timedelta(seconds=float(t)) for t in flat_time_data]
                ## Cộng thời điểm gốc với delta để có giá trị datetime
                datetime_values_flat = [time_origin + delta for delta in time_deltas_flat]
            except Exception as e_time:
                print(f"Lỗi: {e_time}. Giữ giá trị thời gian thô.")
                datetime_values_flat = flat_time_data.tolist()
        else:
            print(f"Giữ giá trị thời gian thô.")
            datetime_values_flat = flat_time_data.tolist()
        
        #Bước 5: Chuyển đổi dữ liệu ngày tháng sang chuỗi định dạng 'YYYY-MM-DD'
        dates_str_for_column = []
        if date_data.ndim == 2 and date_data.shape[0] == len(latitude.flatten()) and date_data.shape[1] >= 3:
            for i in range(date_data.shape[0]):
                year = int(date_data[i, 0])
                month = int(date_data[i, 1])
                day = int(date_data[i, 2])
                dates_str_for_column.append(f"{year}-{month:02d}-{day:02d}")
        else:
            dates_str_for_column = [None] * len(latitude.flatten())

        #Bước 6: Làm phẳng các mảng dữ liệu. Chuyển mảng 2D/3D thành 1D
        lat_flat = latitude.flatten()
        lon_flat = longitude.flatten()
        xco2_flat = xco2_data.flatten() 
        xco2_quality_flag_flat = xco2_quality_flag_data.flatten() 

        # Bước 7: Tạo GeoDataFrame từ các điểm và lọc theo ranh giới Việt Nam
        points_gdf = gpd.GeoDataFrame(
            geometry=gpd.points_from_xy(lon_flat, lat_flat), ## Tạo điểm từ kinh độ và vĩ độ
            crs="EPSG:4326" ## Hệ tọa độ WGS84
        )
        points_gdf['original_index'] = np.arange(len(lat_flat)) ## Lưu chỉ số gốc để truy xuất dữ liệu sau này
        points_gdf['xco2_quality_flag'] = xco2_quality_flag_flat ## Thêm cột cờ chất lượng

        # Bước 8: Đọc shapefile Việt Nam và kiểm tra các điểm nằm trong lãnh thổ Việt Nam
        vietnam_gdf = gpd.read_file(shapefile_path)
        if vietnam_gdf.crs != points_gdf.crs:
            vietnam_gdf = vietnam_gdf.to_crs(points_gdf.crs)

        # Bước 9: Lọc các điểm nằm trong ranh giới Việt Nam
        points_in_vietnam_gdf = gpd.sjoin(points_gdf, vietnam_gdf, how="inner", predicate="within")
        if points_in_vietnam_gdf.empty:
            return
        
        ## Bước 10: Lọc các điểm có chất lượng tốt (xco2_quality_flag == 0)
        good_quality_points_df = points_in_vietnam_gdf[points_in_vietnam_gdf['xco2_quality_flag'] == 0].copy()
        if good_quality_points_df.empty:
            return
        ## Lấy chỉ số gốc của các điểm chất lượng tốt
        filtered_indices = good_quality_points_df['original_index'].values

        # Bước 11: Tạo DataFrame cuối cùng và xuất ra file CSV
        df_data = {
            'latitude': lat_flat[filtered_indices],
            'longitude': lon_flat[filtered_indices],
            'time': [datetime_values_flat[i] for i in filtered_indices],
            'date': [dates_str_for_column[i] for i in filtered_indices],
            'xco2': xco2_flat[filtered_indices],
            'xco2_quality_flag': xco2_quality_flag_flat[filtered_indices], 
        }
        df = pd.DataFrame(df_data)
        df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

In [ ]:
def main():
    # Thư mục chứa file NetCDF4 đầu vào và thư mục đầu ra
    input_folder = "E:\\DownloadData\\co2_nasa\\data\\nc4\\oco2_nc4_2023"  
    output_folder = "E:\\DownloadData\\co2_nasa\\data\\test_csv\\"
    os.makedirs(output_folder, exist_ok=True) 

    # Xử lý tất cả các file .nc4 trong thư mục đầu vào
    nc_files_to_process = glob.glob(os.path.join(input_folder, "*.nc4"))
    for nc_file_path in nc_files_to_process:
        process_single_nc_file(nc_file_path, output_folder, shapefile_path)
    print("Hoàn tất xử lý tất cả các file.")

if __name__ == "__main__":
    main()

In [2]:

# Thư mục chứa các file raw_data
input_folder = "E:\\DownloadData\\co2_nasa\\data\\test_csv"
output_file = "E:\\DownloadData\\co2_nasa\\data\\csv\\2023\\oco2_test\\oco2_vn_qual0_2023.csv"

# Tìm tất cả các file có dạng raw_data_YYYY_MM_DD.csv
all_files = glob.glob(os.path.join(input_folder, "raw_data_*.csv"))

# Danh sách chứa các DataFrame
df_list = []

for file in all_files:
    df = pd.read_csv(file)
    df_list.append(df)

# Gộp tất cả các DataFrame thành một
merged_df = pd.concat(df_list, ignore_index=True)

# Xuất ra file CSV tổng
merged_df.to_csv(output_file, index=False)

print(f"✅ Đã gộp {len(all_files)} file thành công -> {output_file}")


✅ Đã gộp 46 file thành công -> E:\DownloadData\co2_nasa\data\csv\2023\oco2_test\oco2_vn_qual0_2023.csv
